# 01 — Thu thập dữ liệu review từ Google Play

Mục tiêu của notebook này là tạo bộ dữ liệu thô cho toàn bộ khóa luận: review tiếng Việt của Shopee (dùng làm corpus chính cho
Train/Validation/Hold-out) và của Lazada (để làm External Test, kiểm tra mô hình có dùng được cho một app khác hay không).
Dữ liệu thô được lưu một lần dưới dạng JSONL trong `01_data/raw/` và tôi không chỉnh sửa lại các file này về sau, để mọi bước
phía sau đều truy ngược được về đúng một snapshot.

Một vài lựa chọn khi thu thập:

- Tôi dùng thư viện `google-play-scraper` với `lang="vi"`, `country="vn"`, lấy theo thứ tự mới nhất (`Sort.NEWEST`) và lật trang bằng
  `continuation_token`.
- Mốc thời gian của nghiên cứu là T0 = 06/10/2026, là ngày tôi crawl lần đầu. Vì lần crawl lại diễn ra sáng 07/10, tôi bỏ qua mọi
  review có ngày sau 06/10/2026 23:59 để dữ liệu vẫn là ảnh chụp tại T0.
- Thay vì dừng theo số lượng, tôi dừng theo ngày: crawl lùi dần cho tới khi review cũ nhất trong lô đã trước 06/10/2025, tức là
  phủ đủ 12 tháng. Mức trần 120.000 review mỗi lượt chỉ để phòng trường hợp app quá nhiều review.
- Tên người dùng không được lưu: `userName` được thay bằng mã băm 12 ký tự, không lưu ảnh đại diện; tên, số điện thoại, email xuất
  hiện trong nội dung review và phản hồi của sàn được thay bằng `[NAME]`, `[PHONE]`, `[EMAIL]` ngay khi ghi file.

Phần xử lý nằm trong `src/collect/google_play.py`; notebook chỉ gọi các hàm đó và ghi lại kết quả.

### Vì sao Shopee được crawl theo từng mức sao

Lần crawl đầu tiên (06/10) không đạt yêu cầu phủ 12 tháng. Với Shopee, quá trình dừng ở đúng 40.000 review, review cũ nhất mới
tới 12/01/2026, trong khi số review theo tháng vẫn đều (khoảng 3.500–5.500 mỗi tháng). Điều này cho thấy Google Play chỉ cho lật
trang tới khoảng 40.000 review chứ không phải Shopee hết review. Với Lazada, tôi chỉ lấy được 200 review rồi bị trả về rỗng, nhiều
khả năng do chạy ngay sau đợt Shopee nên bị giới hạn tần suất truy cập.

Vì vậy tôi đổi cách làm: Shopee được crawl riêng cho từng mức sao từ 1 đến 5 (`filter_score_with`), mỗi mức dừng theo cùng mốc
ngày, sau đó gộp lại và bỏ trùng theo `review_id`; Lazada được crawl lại sau một khoảng nghỉ. Lúc đầu tôi lo cách này làm phân bố
sao bị lệch, nhưng kết quả ở mục 8 cho thấy cả 5 mức đều đi tới mốc ngày, và khi đối chiếu với lần crawl đầu trên cùng khoảng
12/01–05/10/2026 thì khớp 39.991/39.991 review, phân bố sao gần như trùng nhau (1★ 40,4% / 2★ 3,4% / 3★ 3,3% / 4★ 5,0% / 5★ 47,9%).
Nói cách khác, gộp 5 mức sao lại gần như chính là toàn bộ review của 12 tháng (amendment v1.5). Hai file của lần crawl đầu tôi
vẫn giữ lại làm bằng chứng.

In [1]:
import sys, os, json, time
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir, is_kaggle
from src.utils.seeds import set_seed
from src.collect.google_play import (APPS, T0, T0_END, CUTOFF, SCORES, preview, crawl, crawl_stratified,
                                     merge_dedupe, scrub_file, summarize, accept, write_manifest)
set_seed(42)
# Thu thập cần truy cập Internet nên được thực hiện cục bộ (Kaggle tắt Internet mặc định).
assert not is_kaggle(), 'Giai đoạn 1 được thực hiện cục bộ.'
RAW = input_dir('raw')
TAG = f'T0_{T0.date()}'
print('T0     :', T0.date())
print('T0_END :', T0_END.isoformat())
print('Cutoff :', CUTOFF.isoformat())
print('Apps   :', APPS)

T0     : 2026-10-06
T0_END : 2026-10-06T23:59:59+07:00
Cutoff : 2025-10-06T00:00:00+07:00
Apps   : {'shopee': 'com.shopee.vn', 'lazada': 'com.lazada.android'}


## Kiểm tra thử trước khi crawl toàn bộ

Trước khi chạy thật, tôi lấy thử 50 review mỗi app (không ghi ra đĩa) để chắc rằng mình đang crawl đúng app (xem tiêu đề, nhà phát
triển), biết thư viện trả về những trường nào, và kiểm tra bản ghi sau khi chuyển sang schema của mình đã đúng và đã ẩn tên.

In [2]:
for name, app_id in APPS.items():
    p = preview(app_id, n=50)
    print(f'===== {name} ({app_id})')
    print('Tiêu đề / nhà phát triển / lượt cài:', p['title'], '|', p['developer'], '|', p['installs'])
    print('Số review mẫu:', p['n_fetched'], '| khoảng ngày:', p['date_range'])
    print('Trường của thư viện:', p['library_fields'])
    print('Bản ghi đầu tiên sau ánh xạ:')
    print(json.dumps(p['first_record'], ensure_ascii=False, indent=2))

===== shopee (com.shopee.vn)
Tiêu đề / nhà phát triển / lượt cài: 10.10 Shopee Thương Hiệu | Shopee VN | 100.000.000+
Số review mẫu: 50 | khoảng ngày: ('2026-10-05T11:37:54+00:00', '2026-10-06T00:18:03+00:00')
Trường của thư viện: ['appVersion', 'at', 'content', 'repliedAt', 'replyContent', 'reviewCreatedVersion', 'reviewId', 'score', 'thumbsUpCount', 'userImage', 'userName']
Bản ghi đầu tiên sau ánh xạ:
{
  "review_id": "f57b699f-c4b0-4265-872f-974de5d9097e",
  "text": "zcv 11 t 🌭🥪🥑",
  "rating": 2,
  "review_date": "2026-10-06T00:18:03+00:00",
  "app_version": "3.82.59",
  "thumbs_up": 0,
  "user_hash": "377ffa31815e",
  "reply_content": null,
  "replied_at": null
}


===== lazada (com.lazada.android)
Tiêu đề / nhà phát triển / lượt cài: Lazada 10.10 Sale Đỉnh | Lazada Mobile | 500.000.000+
Số review mẫu: 50 | khoảng ngày: ('2026-09-28T11:16:59+00:00', '2026-10-05T22:45:41+00:00')
Trường của thư viện: ['appVersion', 'at', 'content', 'repliedAt', 'replyContent', 'reviewCreatedVersion', 'reviewId', 'score', 'thumbsUpCount', 'userImage', 'userName']
Bản ghi đầu tiên sau ánh xạ:
{
  "review_id": "33f98773-beba-46ad-af2f-c866cb50be21",
  "text": "ok.",
  "rating": 5,
  "review_date": "2026-10-05T22:45:41+00:00",
  "app_version": null,
  "thumbs_up": 0,
  "user_hash": "1ea847399fab",
  "reply_content": "Cảm ơn bạn, [NAME], đã để lại phản hồi! Chúng tôi luôn trân trọng ý kiến của bạn. Nếu bạn có bất kỳ vấn đề hay thắc mắc nào, hãy liên hệ với đội ngũ Hỗ trợ tại https://helpcenter.lazada.vn/s/faq để được trợ giúp nhé! Chúc bạn có trải nghiệm tuyệt vời!",
  "replied_at": "2026-10-05T22:55:44+00:00"
}


Tôi chỉ chạy crawl toàn bộ sau khi đã xem kết quả kiểm tra thử ở trên; biến `CONFIRMED` đánh dấu bước xác nhận này.

In [3]:
CONFIRMED = True   # đã xác nhận schema và định danh ứng dụng từ kiểm tra sơ bộ
assert CONFIRMED, 'Kiểm tra sơ bộ chưa được xác nhận.'

## Ẩn danh hóa lại hai file của lần crawl đầu

Hai file của lần crawl 06/10 được tạo trước khi tôi thêm bước ẩn danh hóa, nên tôi xử lý chúng sang hai file mới `*_scrubbed.jsonl`
và giữ nguyên file gốc.

In [4]:
scrub_runs = []
for name in APPS:
    src_path = os.path.join(RAW, name, f'{name}_reviews_{TAG}.jsonl')
    dst_path = src_path.replace('.jsonl', '_scrubbed.jsonl')
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        scrub_runs.append({**scrub_file(src_path, dst_path), **summarize(dst_path)})
        print(json.dumps(scrub_runs[-1], ensure_ascii=False, indent=2))

{
  "in_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\shopee\\shopee_reviews_T0_2026-10-06.jsonl",
  "out_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\shopee\\shopee_reviews_T0_2026-10-06_scrubbed.jsonl",
  "n_reviews": 40000,
  "n_pii_replacements": 155,
  "min_review_date": "2026-01-12T09:37:48+00:00",
  "max_review_date": "2026-10-05T14:36:52+00:00",
  "rating_counts": {
    "1": 16156,
    "2": 1359,
    "3": 1317,
    "4": 2017,
    "5": 19151
  },
  "app_version_null_rate": 0.2666,
  "sha256": "4a1ba30eb8ecba5c867b936171bbf73db0589100d3dbe905615ea4fb57b1681b"
}
{
  "in_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\lazada\\lazada_reviews_T0_2026-10-06.jsonl",
  "out_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\lazada\\lazada_reviews_T0_2026-10-06_scrubbed.jsonl",
  "n_reviews": 200,
  "n_pii_replacements": 103,
  "min_review_date": "2026-09-07T08:56:22+00:00",
  "max_review_date": "20

## Crawl Shopee theo từng mức sao

Mỗi mức sao được ghi vào một file riêng (`shopee_stratified_<T0>_score<k>.jsonl`) kèm một file thống kê `.stats.json`, giữa các mức
nghỉ 60 giây. Tôi viết hàm sao cho có thể chạy tiếp nếu bị ngắt giữa chừng: mức nào đã có file hợp lệ thì giữ nguyên, file rỗng
hoặc hỏng thì crawl lại. Sau đó 5 file được gộp và bỏ trùng theo `review_id`.

In [5]:
shopee_dir = os.path.join(RAW, 'shopee')
strata = crawl_stratified(APPS['shopee'], shopee_dir, f'shopee_stratified_{TAG}', scores=SCORES,
                          pause_between=60, cap=120000, batch=200, sleep=1.0)
for s in strata:
    s['hit_cap'] = s['stop_reason'] == 'safety_cap'
    s['accepted'] = accept(s)
shopee_merged = os.path.join(shopee_dir, f'shopee_reviews_stratified_{TAG}.jsonl')
if os.path.exists(shopee_merged):   # tệp gộp của lượt trước được giữ nguyên
    print('Tệp gộp đã tồn tại, giữ nguyên:', shopee_merged)
    merge = {'merged_from': [os.path.basename(s['out_path']) for s in strata], 'skipped_existing': True}
else:
    merge = merge_dedupe([s['out_path'] for s in strata], shopee_merged)
shopee_run = {'app': 'shopee', 'package_id': APPS['shopee'], 'mode': 'stratified_by_rating',
              'strata': strata, 'merge': merge, **summarize(shopee_merged)}
shopee_run['n_excluded_after_t0'] = sum(s.get('n_excluded_after_t0') or 0 for s in strata)
shopee_run['hit_cap'] = any(s['hit_cap'] for s in strata)
shopee_run['accepted'] = accept(shopee_run)
print(json.dumps({k: v for k, v in shopee_run.items() if k != 'strata'}, ensure_ascii=False, indent=2))

===== com.shopee.vn score=1 → C:\Workspace\kltn-vn-app-review-requirements\01_data\raw\shopee\shopee_stratified_T0_2026-10-06_score1.jsonl


  com.shopee.vn[1 sao]: 5000 reviews, oldest=2026-07-22T14:02:17+00:00


  com.shopee.vn[1 sao]: 10000 reviews, oldest=2026-05-02T15:29:50+00:00


  com.shopee.vn[1 sao]: 15000 reviews, oldest=2026-01-29T01:57:12+00:00


  com.shopee.vn[1 sao]: 20000 reviews, oldest=2025-11-13T12:25:39+00:00


===== com.shopee.vn score=2 → C:\Workspace\kltn-vn-app-review-requirements\01_data\raw\shopee\shopee_stratified_T0_2026-10-06_score2.jsonl


... (5 outputs left out)

  com.shopee.vn[5 sao]: 15000 reviews, oldest=2026-03-06T08:14:27+00:00


  com.shopee.vn[5 sao]: 20000 reviews, oldest=2026-01-05T13:36:53+00:00


  com.shopee.vn[5 sao]: 25000 reviews, oldest=2025-11-07T06:45:05+00:00


{
  "app": "shopee",
  "package_id": "com.shopee.vn",
  "mode": "stratified_by_rating",
  "merge": {
    "merged_from": [
      "shopee_stratified_T0_2026-10-06_score1.jsonl",
      "shopee_stratified_T0_2026-10-06_score2.jsonl",
      "shopee_stratified_T0_2026-10-06_score3.jsonl",
      "shopee_stratified_T0_2026-10-06_score4.jsonl",
      "shopee_stratified_T0_2026-10-06_score5.jsonl"
    ],
    "n_in": 57200,
    "n_out": 57200,
    "n_duplicates": 0,
    "out_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\shopee\\shopee_reviews_stratified_T0_2026-10-06.jsonl"
  },
  "n_reviews": 57200,
  "min_review_date": "2025-09-12T01:41:21+00:00",
  "max_review_date": "2026-10-06T00:18:03+00:00",
  "rating_counts": {
    "1": 22600,
    "2": 2000,
    "3": 2000,
    "4": 3000,
    "5": 27600
  },
  "app_version_null_rate": 0.2626,
  "sha256": "d8115c1fd7d93812bc8c4d480889ff14e143b2ff63f9717d307bc76149657a72",
  "n_excluded_after_t0": 0,
  "hit_cap": false,
  "accepted": 

## Crawl lại Lazada

Lazada được crawl theo thứ tự mới nhất, không tách theo sao, và tôi để nghỉ 10 phút sau đợt Shopee cho chắc không bị chặn như lần
trước. Kết quả ghi vào file mới có hậu tố `_r2`.

In [6]:
LAZADA_PAUSE = 600   # giây
print(f'Khoảng nghỉ {LAZADA_PAUSE} giây trước khi thu thập Lazada.', flush=True)
time.sleep(LAZADA_PAUSE)
laz_out = os.path.join(RAW, 'lazada', f'lazada_reviews_{TAG}_r2.jsonl')
stats = crawl(APPS['lazada'], laz_out, cutoff=CUTOFF, cap=120000, batch=200, sleep=1.0)
lazada_run = {'app': 'lazada', 'package_id': APPS['lazada'], 'mode': 'newest', **stats, **summarize(laz_out)}
lazada_run['hit_cap'] = stats['stop_reason'] == 'safety_cap'
lazada_run['accepted'] = accept(lazada_run)
print(json.dumps(lazada_run, ensure_ascii=False, indent=2))

Khoảng nghỉ 600 giây trước khi thu thập Lazada.


{
  "app": "lazada",
  "package_id": "com.lazada.android",
  "mode": "newest",
  "app_id": "com.lazada.android",
  "filter_score_with": null,
  "out_path": "C:\\Workspace\\kltn-vn-app-review-requirements\\01_data\\raw\\lazada\\lazada_reviews_T0_2026-10-06_r2.jsonl",
  "n_reviews": 3600,
  "n_batches": 18,
  "n_pii_replacements": 1940,
  "n_excluded_after_t0": 0,
  "t0_end": "2026-10-06T23:59:59+07:00",
  "stop_reason": "reached_cutoff",
  "started_utc": "2026-10-07T00:54:18.471129+00:00",
  "finished_utc": "2026-10-07T00:54:45.724678+00:00",
  "cutoff": "2025-10-06T00:00:00+07:00",
  "cap": 120000,
  "batch": 200,
  "min_review_date": "2025-09-26T06:06:47+00:00",
  "max_review_date": "2026-10-05T22:45:41+00:00",
  "rating_counts": {
    "1": 2165,
    "2": 125,
    "3": 141,
    "4": 134,
    "5": 1035
  },
  "app_version_null_rate": 0.3419,
  "sha256": "c7c882890de91be22b1e5789f0b619dedbe04316cae4f2e776ddecf7afddef85",
  "hit_cap": false,
  "accepted": true
}


## Kết quả và mức độ đầy đủ

Tiêu chí tôi đặt ra là review cũ nhất phải ở trước hoặc đúng 06/10/2025 với cả hai app. Kết quả bên dưới cho thấy cả 5 mức sao
của Shopee và Lazada đều dừng vì đã đi qua mốc ngày, không mức nào chạm trần. Shopee sau khi gộp có 57.200 review, Lazada 3.600.
Không có review nào bị loại vì sau T0 — review mới nhất chỉ tới sáng 06/10, có vẻ Google Play hiển thị review trễ khoảng một ngày.

In [7]:
def verdict(r):
    if r['accepted']:
        return 'đạt (tới cutoff)'
    if r['hit_cap']:
        return 'không đạt: chạm trần an toàn trước cutoff'
    return 'không đạt: Google Play ngừng trả dữ liệu trước cutoff'

print('Shopee theo mức đánh giá')
for s in strata:
    print(f"  {s['filter_score_with']} sao | n={s['n_reviews']:>6} | min={s['min_review_date']} | dừng={s.get('stop_reason')} "
          f"| chạm trần={s['hit_cap']} | loại sau T0={s.get('n_excluded_after_t0')} | tiếp tục={s['resumed']} | {verdict(s)}")
print('  Gộp:', {k: merge[k] for k in ('n_in', 'n_out', 'n_duplicates', 'skipped_existing') if k in merge})
runs = [shopee_run, lazada_run]
print('Theo ứng dụng')
for r in runs:
    print(f"  {r['app']:7s} | n={r['n_reviews']} | min={r['min_review_date']} | max={r['max_review_date']} "
          f"| loại sau T0={r['n_excluded_after_t0']} | thiếu app_version={r['app_version_null_rate']:.1%} "
          f"| mức đánh giá={r['rating_counts']} | {verdict(r)}")
manifest_path = os.path.join(input_dir('manifests'), f'gd1_crawl_manifest_{TAG}_r2.json')
manifest = write_manifest(runs + [{'post_process_scrub': scrub_runs}], manifest_path)
print('Manifest:', manifest_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

Shopee theo mức đánh giá
  1 sao | n= 22600 | min=2025-10-02T09:30:05+00:00 | dừng=reached_cutoff | chạm trần=False | loại sau T0=0 | tiếp tục=False | đạt (tới cutoff)
  2 sao | n=  2000 | min=2025-09-16T07:38:29+00:00 | dừng=reached_cutoff | chạm trần=False | loại sau T0=0 | tiếp tục=False | đạt (tới cutoff)
  3 sao | n=  2000 | min=2025-09-12T01:41:21+00:00 | dừng=reached_cutoff | chạm trần=False | loại sau T0=0 | tiếp tục=False | đạt (tới cutoff)
  4 sao | n=  3000 | min=2025-09-14T16:25:27+00:00 | dừng=reached_cutoff | chạm trần=False | loại sau T0=0 | tiếp tục=False | đạt (tới cutoff)
  5 sao | n= 27600 | min=2025-10-03T06:27:25+00:00 | dừng=reached_cutoff | chạm trần=False | loại sau T0=0 | tiếp tục=False | đạt (tới cutoff)
  Gộp: {'n_in': 57200, 'n_out': 57200, 'n_duplicates': 0}
Theo ứng dụng
  shopee  | n=57200 | min=2025-09-12T01:41:21+00:00 | max=2026-10-06T00:18:03+00:00 | loại sau T0=0 | thiếu app_version=26.3% | mức đánh giá={'1': 22600, '2': 2000, '3': 2000, '4': 3000, '

## Kiểm tra trường phản hồi của sàn

Tôi in một bản ghi có `reply_content` của mỗi app để xem thời điểm phản hồi (`replied_at`, giờ UTC) có hợp lý so với thời điểm
review không, và tên người dùng trong phản hồi đã được thay bằng `[NAME]` chưa.

In [8]:
for path in (shopee_merged, laz_out):
    with open(path, encoding='utf-8') as f:
        rec = next((r for r in map(json.loads, f) if r.get('reply_content')), None)
    print(f'===== {os.path.basename(path)}')
    print(json.dumps(rec, ensure_ascii=False, indent=2) if rec else '(không có bản ghi nào có reply_content)')

===== shopee_reviews_stratified_T0_2026-10-06.jsonl
{
  "review_id": "565bb4e5-44b5-4a91-b2df-4905a8de90ff",
  "text": "Bây giờ o ép người dùng quá rồi, phải đăng ký ShopeeVip thì mới có mã giảm nhiều mà lại không cho đăng ký cứ báo ShopeeVip không khả dụng. Thôi gỡ app luôn đi. Lọc người này lọc người kia. App hèn hạ chưa từng thấy. Tôi qua TikTok mua luôn cho khoẻ người.",
  "rating": 1,
  "review_date": "2026-10-05T23:26:39+00:00",
  "app_version": "3.82.59",
  "thumbs_up": 0,
  "user_hash": "375869a8b81d",
  "reply_content": "Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hiệu quả nhất, bạn vui lòng truy cập ứng dụng Shopee >> trang Tôi >> Trung tâm trợ giúp >> Kéo xuống dưới cùng màn hình và nhấn Gọi tổng đài Shopee miễn phí.",
  "replied_at": "2026-10-06T09:40:15+00:00"
}
===== lazada_reviews_T0_2026-10-06_r2.jsonl
{
  "review_id": "33f98773-beba-46ad-af2f-c866cb50be21",

## Lưu trữ dữ liệu

Các file trong `01_data/raw/` chỉ được giữ trên máy làm snapshot gốc, không đưa lên Git và cũng không đưa lên Kaggle. Lý do là ở
Giai đoạn 2, khi kiểm tra kỹ phần ẩn danh hóa, tôi phát hiện luật thay tên dùng lúc crawl còn sót khoảng 1.100 tên người dùng
trong phản hồi của sàn (chủ yếu ở Lazada). Bộ dữ liệu tôi đưa lên Kaggle (dạng Dataset riêng tư) cho các notebook sau là bản
trung gian đã ẩn danh hóa lại ở Giai đoạn 2: `01_data/interim/{shopee,lazada}_clean_T0_2026-10-06.jsonl` cùng manifest.

## Những hạn chế tôi ghi nhận

- Shopee được crawl theo từng mức sao, nhưng như đã phân tích ở trên, phân bố thu được gần như trùng với phân bố tự nhiên. Hạn chế
  còn lại là dữ liệu chỉ gồm những review Google Play còn hiển thị tại thời điểm crawl; review đã bị xóa hoặc ẩn thì không có.
- Lazada có tỷ lệ review 1★ cao hơn hẳn Shopee (khoảng 60% so với 40%). Đây là đặc điểm thật của dữ liệu, không phải do cách crawl,
  nhưng tôi cần nhớ điều này khi so sánh kết quả mô hình giữa hai app.
- Khá nhiều review không có `app_version` (Shopee 26,3%, Lazada 34,2%), nên khi phân tích theo thời gian tôi sẽ dựa chủ yếu vào ngày
  review.
- Việc ẩn danh hóa dựa trên luật nên khó bắt hết mọi trường hợp; phần kiểm tra bổ sung được làm ở Giai đoạn 2.